In [ ]:
# Import libraries
import pandas as pd # do i need this?
import torch
from torch.utils.data import DataLoader, TensorDataset
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import pyreadr # for reading Rdata files
from transformers import BertTokenizer
from sklearn.model_selection import train_test_split

In [282]:
# Import training and testing data
claims_clean_primary = pyreadr.read_r("../data/claims-clean-primary.RData")
# extract dataframe from claims objects
df_claims_clean = claims_clean_primary["claims_clean"]
# Add labels column for training data
df_claims_clean['labels'] = (df_claims_clean['bclass'] == "Relevant claim content").astype(int)
# split df_claims_clean into training and testing data
df_claims_train, df_claims_test = train_test_split(
    df_claims_clean,
    test_size = 0.2,
    random_state = 111625
)
# Reset indices to avoid errors in batch encoding
df_claims_train = df_claims_train.reset_index(drop=True)
df_claims_test = df_claims_test.reset_index(drop=True)

# save testing dataset
df_claims_test.to_csv("../data/claims_test.csv")

In [162]:
# preprocessing - tokenize data
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

claims_train_tokenized = tokenizer.batch_encode_plus(
    df_claims_train["text_clean"], padding = True, truncation = True,
    return_tensors = "pt"
)

claims_test_tokenized = tokenizer.batch_encode_plus(
    df_claims_test["text_clean"], padding = True, truncation = True,
    return_tensors = "pt"
)

In [191]:
# Define dataloader for training and testing
train_dataset = TensorDataset(claims_train_tokenized['input_ids'].type(torch.float32),
                              torch.tensor(df_claims_train['labels']).type(torch.float32))
test_dataset = TensorDataset(claims_test_tokenized['input_ids'].type(torch.float32),
                              torch.tensor(df_claims_test['labels']).type(torch.float32))


train_dataloader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_dataloader = DataLoader(test_dataset, batch_size=64, shuffle=True)

In [ ]:
# Define model architecture
class LSTMModel(nn.Module):
    def __init__(self, vocab_size, embedding_size, hidden_size, num_layers, output_size):
        super(LSTMModel, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_size)
        self.lstm = nn.LSTM(embedding_size, hidden_size, num_layers, batch_first = True,
                            bidirectional=True)
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, text):
        embedded = self.embedding(text)
        output, (hn, cn) = self.lstm(embedded)
        # pass the last timepoint of the hidden layer into linear layer
        out = self.fc(hn[-1])
        return out

In [452]:
# Define training function for NN
# LSTM model with input, x size output, 1 hidden layers
# and 2 output classes (not relevant or relevant)
vocab_size = tokenizer.vocab_size
embedding_size = 128
hidden_size = 64
model = LSTMModel(vocab_size, embedding_size, hidden_size, 1, 2)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr = 0.0005)

def train(model, num_epochs, criterion, optimizer):
    for epoch in range(num_epochs):
        # enumerate batches
        for i, (batch, label) in enumerate(train_dataloader):
            # Reset optimizer gradient
            optimizer.zero_grad()
            
            # Cast to long to avoid Pytorch error
            batch = batch.long()
            label = label.long()
            
            outputs_train = model(batch)

            # calculate loss
            loss = criterion(outputs_train, label)
            # get gradients
            loss.backward()
            # update parameters
            optimizer.step()
            
        print(f"Epoch: {epoch + 1}, Loss: {loss.item():.4f}")

In [453]:
# Train NN
train(model, 20, criterion, optimizer)

Epoch: 1, Loss: 0.6587
Epoch: 2, Loss: 0.6191
Epoch: 3, Loss: 0.6243
Epoch: 4, Loss: 0.3942
Epoch: 5, Loss: 0.4475
Epoch: 6, Loss: 0.5930
Epoch: 7, Loss: 0.3725
Epoch: 8, Loss: 0.2510
Epoch: 9, Loss: 0.3914
Epoch: 10, Loss: 0.1549
Epoch: 11, Loss: 0.1840
Epoch: 12, Loss: 0.1972
Epoch: 13, Loss: 0.1679
Epoch: 14, Loss: 0.1696
Epoch: 15, Loss: 0.1473
Epoch: 16, Loss: 0.0775
Epoch: 17, Loss: 0.0878
Epoch: 18, Loss: 0.1296
Epoch: 19, Loss: 0.1215
Epoch: 20, Loss: 0.2424


In [416]:
# Save trained model details
torch.save(model.state_dict(), "../results/LSTM_NN_predictive_model.pt")

In [454]:
# Test NN
def test(model):
    correct = 0
    total = 0
    model.eval()
    with torch.no_grad():
        for i, (batch, label) in enumerate(test_dataloader):
            batch = batch.long()
            label = label.long()
            outputs_test = model(batch)
            probs = F.log_softmax(outputs_test, dim = 1)
            pred_class = torch.argmax(probs, dim = 1)
            correct += (pred_class == label).sum()
            total += label.size(0)
    accuracy = 100 * correct.item() / total
    print(f"Accuracy: {accuracy}%")


In [455]:
test(model)

Accuracy: 74.53271028037383%


In [ ]:
# Define model architecture for extended LSTM with an extra linear layer
class LSTMModel_extended(nn.Module):
    def __init__(self, vocab_size, embedding_size, hidden_size, num_layers, output_size):
        super(LSTMModel_extended, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_size)
        self.lstm = nn.LSTM(embedding_size, hidden_size, num_layers, batch_first = True,
                            bidirectional=True)
        self.fc = nn.Linear(hidden_size, hidden_size)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_size, output_size)

    def forward(self, text):
        embedded = self.embedding(text)
        output, (hn, cn) = self.lstm(embedded)
        # pass the last timepoint of the hidden layer into linear layer
        out = self.fc(hn[-1])
        out = self.relu(out)
        out = self.fc2(out)
        return out

In [ ]:
vocab_size = tokenizer.vocab_size
embedding_size = 128
hidden_size = 64
model_extended_two = LSTMModel_extended(vocab_size, embedding_size, hidden_size, 1, 2)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model_extended_two.parameters(), lr = 0.001)

def train_extended(model, num_epochs, criterion, optimizer):
    for epoch in range(num_epochs):
        losses = 0
        # enumerate batches
        for i, (batch, label) in enumerate(train_dataloader):
            # Reset optimizer gradient
            optimizer.zero_grad()
            
            # Cast to long to avoid Pytorch error
            batch = batch.long()
            label = label.long()
            
            outputs_train = model(batch)

            # calculate loss
            loss = criterion(outputs_train, label)
            losses += loss.item()
            # get gradients
            loss.backward()
            # update parameters
            optimizer.step()
            
        print(f"Epoch: {epoch + 1}, Loss: {losses / len(train_dataloader.dataset):.4f}")

train_extended(model_extended_two, 20, criterion, optimizer)

Epoch: 1, Loss: 0.0108
Epoch: 2, Loss: 0.0099
Epoch: 3, Loss: 0.0086
Epoch: 4, Loss: 0.0070
Epoch: 5, Loss: 0.0056
Epoch: 6, Loss: 0.0041
Epoch: 7, Loss: 0.0034
Epoch: 8, Loss: 0.0030
Epoch: 9, Loss: 0.0029
Epoch: 10, Loss: 0.0028
Epoch: 11, Loss: 0.0026
Epoch: 12, Loss: 0.0026
Epoch: 13, Loss: 0.0026
Epoch: 14, Loss: 0.0025
Epoch: 15, Loss: 0.0027
Epoch: 16, Loss: 0.0025
Epoch: 17, Loss: 0.0024
Epoch: 18, Loss: 0.0024
Epoch: 19, Loss: 0.0023
Epoch: 20, Loss: 0.0022


In [495]:
test(model_extended_two)

Accuracy: 76.40186915887851%


In [503]:
# Save trained model details
torch.save(model_extended_two.state_dict(), "../results/LSTM_NN_predictive_model_extended.pt")